# M5: benchmark generation and scoring (PLAN §7 M5; D-050, D-086)

**Notebook settings:** Accelerator **GPU T4 x2** · Internet **on** · Secrets: **HF_TOKEN**. Run as **Save Version → Save & Run All**.

Set `CONFIGS` in the first code cell, one session per row:
- **Session A:** `["smoke", "tier2_qwen2.5-0.5b", "tier2_llama3.2-1b"]`.
  - `smoke` is DC-11: the whole pipeline, now including `generate` and `score`, in < 15 min.
  - After smoke the GPU tests run: DC-13 (generation resume on real vLLM) and the WildGuard check (D-038).
- **Session B:** `["tier1_llama3.1-8b"]`. **Session C:** `["tier1_mistral-7b"]`. **Session D (and E if paused):** `["tier1_gemma3-4b"]` (fp32, D-005).

Per config, the notebook:
1. restores the trained endpoints and any earlier generations and labels from the private store;
2. runs the pipeline through `score`:
   - Training is skipped, because every endpoint has `done.json`.
   - Each checkpoint is merged to ephemeral disk and served by one vLLM engine for all three benchmarks, then the merged copy is deleted (D-050).
   - Prompt files are fetched at run time from pinned commits and never stored (D-086).
3. at `DEADLINE_H` stops cleanly between shards (exit 76). Run the same session again and it resumes.
4. re-runs `generate,score` (both must be cache hits), then syncs generations, labels, manifests and results to the private store.

WGM is scored only once the frozen topic map is pinned (`bench.wgm.topic_map_sha256`, D-043). Until then `score` lists it in `skipped.json`. Only aggregates are printed: no prompt and no generated text.

In [ ]:
REPO = "SGupta-4/Reference-Based-Bias-Detection-Implementation"
REF = "main"  # a branch, a tag, or a FULL 40-character commit SHA (short SHAs fail)
WORK = "/kaggle/working/rbbd"
CONFIGS = ["smoke", "tier2_qwen2.5-0.5b", "tier2_llama3.2-1b"]  # A; B: tier1_llama3.1-8b; C: tier1_mistral-7b; D: tier1_gemma3-4b
GPU_TESTS = True     # after smoke: DC-13 generation resume + WildGuard (D-038); needs the smoke WGM generations
DEADLINE_H = 11.0    # generation stops cleanly between shards after this many session hours
import os, time
SESSION_T0 = time.time()
os.environ["RBBD_DEADLINE_UNIX"] = str(SESSION_T0 + DEADLINE_H * 3600)

In [ ]:
# Clone the repo at REF (branch, tag or commit SHA). If the repo is private, attach the
# GITHUB_TOKEN secret: it is passed to git as an HTTP header through environment
# variables, never in the URL, argv or output.
import base64, datetime, os, shutil, subprocess
from kaggle_secrets import UserSecretsClient

os.environ.setdefault(
    "RBBD_SESSION_ID", datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
)
RBBD_SESSION_ID = os.environ["RBBD_SESSION_ID"]  # also expands $RBBD_SESSION_ID in ! lines
git_env = {**os.environ, "GIT_TERMINAL_PROMPT": "0"}  # fail instead of prompting for a password
try:
    _gh = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    _gh = None
if _gh:
    _basic = base64.b64encode(f"x-access-token:{_gh}".encode()).decode()
    git_env.update(GIT_CONFIG_COUNT="1", GIT_CONFIG_KEY_0="http.https://github.com/.extraheader",
                   GIT_CONFIG_VALUE_0=f"AUTHORIZATION: basic {_basic}")
del _gh

shutil.rmtree(WORK, ignore_errors=True)
os.makedirs(WORK)
for args in (["init", "-q"], ["remote", "add", "origin", f"https://github.com/{REPO}.git"],
             ["fetch", "-q", "--depth", "1", "origin", REF], ["checkout", "-q", "FETCH_HEAD"]):
    done = subprocess.run(["git", *args], cwd=WORK, env=git_env, capture_output=True, text=True)
    if done.returncode and args[0] == "fetch" and "GIT_CONFIG_COUNT" in git_env:
        # An expired or revoked token gets HTTP 401 even on a public repo (B-025): retry
        # without it. git's stderr never contains the header value.
        print("GITHUB_TOKEN was rejected (expired or revoked?); retrying without it")
        git_env = {k: v for k, v in git_env.items() if not k.startswith("GIT_CONFIG_")}
        done = subprocess.run(["git", *args], cwd=WORK, env=git_env, capture_output=True, text=True)
    if done.returncode:
        raise RuntimeError(f"git {args[0]} failed: {done.stderr[-500:]}")
del git_env
os.chdir(WORK)
print("commit", subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip(),
      "| session", os.environ["RBBD_SESSION_ID"])

In [ ]:
# B-032 / D-087: the Kaggle image now runs Python 3.13 and vllm 0.10.1.1 has no 3.13 wheels.
# The pinned stack goes into a Python 3.12 venv on ephemeral disk; putting its bin/ first on
# PATH makes every `!python ...` line (and the stage subprocesses) use it. This kernel only
# imports rbbd's light modules (config, runner, manifests) from src/.
import os
EPH = next((p for p in ("/kaggle/tmp", "/kaggle/temp") if os.path.isdir(p)), "/tmp")
VENV = f"{EPH}/rbbd_py312"
os.environ.update(UV_CACHE_DIR=f"{EPH}/uv_cache", UV_LINK_MODE="copy")
!pip install -q uv==0.8.17 2>&1 | tail -2
!uv venv -q --python 3.12 {VENV}
os.environ["VIRTUAL_ENV"] = VENV
os.environ["PATH"] = f"{VENV}/bin:" + os.environ["PATH"]
!uv pip install -q -e ".[train,bench,dev]" 2>&1 | tail -5
!uv pip freeze > /kaggle/working/pip_freeze_m5.txt
# Preflight (B-012, D-049): stop here, not inside a stage, if the pinned stack did not install.
import subprocess
_pf = subprocess.run(["python", "-c", "import sys, torch, torchvision, transformers, vllm; "
                      "print('preflight ok', sys.version.split()[0], sys.executable, torch.__version__, "
                      "torchvision.__version__, transformers.__version__, vllm.__version__)"],
                     capture_output=True, text=True)
print(_pf.stdout.strip(), _pf.stderr[-1500:])
if _pf.returncode:
    raise RuntimeError("preflight failed: the Python 3.12 venv stack did not install (see above)")

In [ ]:
# Copy Kaggle Secrets into the environment (presence flags only are printed) and keep the
# HF cache on ephemeral disk so it never counts against /kaggle/working (D-029/D-041).
import sys
sys.path.insert(0, f"{WORK}/src")  # the editable install is not visible to this already-running kernel
from rbbd.utils.env import ephemeral_dir, load_kaggle_secrets

_secrets = load_kaggle_secrets(["HF_TOKEN"])
print(_secrets)
if not _secrets["HF_TOKEN"]:
    # Stop here instead of failing later on every gated download (B-022).
    raise RuntimeError("HF_TOKEN secret not attached: Add-ons -> Secrets -> enable HF_TOKEN "
                       "for this notebook, then run again")
os.environ["HF_HOME"] = str(ephemeral_dir() / "hf")
ENV_DIR = f"/kaggle/working/artifacts/env/{os.environ['RBBD_SESSION_ID']}"
os.makedirs(ENV_DIR, exist_ok=True)

In [ ]:
# One config at a time: restore, generate + score, cache check, sync. Aggregates only.
import csv, glob, json, shutil, subprocess
from collections import defaultdict
from pathlib import Path
from rbbd import config as _config, runner
from rbbd.utils import manifest as mf
ROOT = Path("/kaggle/working/artifacts")
SUMMARY = {}
BENCHES = ("dt", "toxigen", "wgm")

def manifest(cfg, stage):
    return mf.read(mf.manifest_path(ROOT, stage, runner.stage_run_keys(cfg)[stage]))

def stage_file(cfg, stage, name):
    return ROOT / stage / runner.stage_run_keys(cfg)[stage] / name

def show_logs(cfg):
    # generate-one logs: progress, timing and tracebacks only (no prompt or generated text).
    for log in sorted(glob.glob(str(stage_file(cfg, "generate", "logs") / "*.log"))):
        print("----", Path(log).name); print("".join(open(log).readlines()[-15:]))

def score_summary(path):
    # Mean score and mean Δ vs ref per (bench, regime, checkpoint) over its units.
    acc = defaultdict(lambda: [0.0, 0.0, 0])
    for r in csv.DictReader(open(path)):
        if r["unit_type"] == "group_from_topic":
            continue
        a = acc[(r["bench"], r["regime"], r["ckpt"])]
        a[0] += float(r["score"]); a[1] += float(r["delta"] or 0); a[2] += 1
    return {"/".join(k): {"mean_score": round(s / n, 4), "mean_delta": round(d / n, 4), "units": n}
            for k, (s, d, n) in sorted(acc.items())}

for NAME in CONFIGS:
    CFG = f"configs/{NAME}.yaml"
    cfg = _config.load(CFG)
    SLUG, RUN = cfg["models"][0]["slug"], cfg["run_name"]
    out = SUMMARY.setdefault(NAME, {"config_hash": cfg.hash})
    print(f"===== {NAME} ({SLUG}) =====")
    if NAME != "smoke":
        !python -m rbbd.cli restore --path train/{SLUG}
    for b in BENCHES:  # earlier sessions' finished shards and labels (resume)
        !python -m rbbd.cli restore --path generations/{b}/{SLUG}
        !python -m rbbd.cli restore --path scores/{b}/{SLUG}
    if NAME == "smoke":
        stages = "sentences,ftdata,train,embed,deltab,generate,score"
    else:
        stages = "sentences,ftdata,train,generate,score"
        !python -m rbbd.cli run --config {CFG} --stages sentences,ftdata
        # Never train here: every endpoint must already be done (as in M4).
        from rbbd.finetune import sft
        jobs = sft.plan_jobs(cfg, ROOT, list(manifest(cfg, "ftdata").output_hashes))
        missing = [str(j.out_dir) for j in jobs if not (j.out_dir / "done.json").exists()]
        if missing:
            raise RuntimeError(f"{NAME}: endpoints not found after restore (train keys changed?): {missing}")
    t0 = time.time()
    !python -m rbbd.cli run --config {CFG} --stages {stages}
    out["pipeline_wall_s"] = round(time.time() - t0, 1)
    gen_m, sc_m = manifest(cfg, "generate"), manifest(cfg, "score")
    out["generate_complete"] = bool(gen_m and gen_m.complete)
    out["score_complete"] = bool(sc_m and sc_m.complete)
    if not out["generate_complete"] or not out["score_complete"]:
        out["paused"] = bool(gen_m and gen_m.error and "paused" in gen_m.error)
        if not out["paused"]:
            show_logs(cfg)
    if out["generate_complete"]:
        timing = json.loads(stage_file(cfg, "generate", "timing.json").read_text())
        shutil.copy(stage_file(cfg, "generate", "timing.json"), f"{ENV_DIR}/generate_timing_{NAME}.json")
        per_bench = defaultdict(float)
        for t in timing:
            per_bench[t["bench"]] += t["seconds"]
        per_ckpt = defaultdict(float)
        for t in timing:
            per_ckpt[f'{t["spectrum"]}/{t["ckpt"]}'] += t["seconds"]
        out["dc14"] = {"jobs": len(timing), "seconds_by_bench": {k: round(v) for k, v in per_bench.items()},
                       "seconds_by_ckpt": {k: round(v) for k, v in per_ckpt.items()},
                       "max_processes_per_job": max(t["processes"] for t in timing)}
        index = json.loads(stage_file(cfg, "generate", "index.json").read_text())
        done = [json.loads((ROOT / r["dir"] / "done.json").read_text()) for r in index]
        out["generation"] = {"jobs": len(index), "items": sum(d["n_items"] for d in done),
                             "skipped_too_long": sum(len(d["skipped_too_long"]) for d in done),
                             "render_modes": {m: sum(d["render_modes"].get(m, 0) for d in done)
                                              for m in {m for d in done for m in d["render_modes"]}}}
    if out["score_complete"]:
        out["skipped"] = json.loads(stage_file(cfg, "score", "skipped.json").read_text())
        out["scores"] = score_summary(stage_file(cfg, "score", "bench_scores.csv"))
        # Cache check: both stages must be cache hits now.
        r = subprocess.run(["python", "-m", "rbbd.cli", "run", "--config", CFG, "--stages", "generate,score"],
                           capture_output=True, text=True)
        log = r.stdout + r.stderr
        out["cache_hits"] = {s: f"cache hit: stage {s}" in log for s in ("generate", "score")}
    res = Path(WORK) / "results" / RUN
    if res.is_dir():
        shutil.copytree(res, ROOT / "results" / RUN, dirs_exist_ok=True)
    paths = ["manifests", "generate", "score", f"results/{RUN}"]
    paths += [f"{d}/{b}/{SLUG}" for d in ("generations", "scores") for b in BENCHES]
    if NAME == "smoke":
        paths += ["sentences", "ftdata", "embed", "deltab", f"embeddings/{SLUG}"]
    for path in paths:
        if (ROOT / path).exists():
            !python -m rbbd.cli sync --path {path}
    print(json.dumps(out, indent=1, default=str))
    if NAME == "smoke" and GPU_TESTS and out["generate_complete"]:
        # DC-13 (vLLM resume) and D-038 (WildGuard on fixed pairs + 20 smoke WGM samples).
        !python -m pytest -q -m gpu tests/gpu/test_generate_gpu.py -k "resume or wildguard" 2>&1 | tail -15
        p = Path(ENV_DIR) / "m5_generate_gpu.json"
        out["gpu_tests"] = json.loads(p.read_text()) if p.exists() else "no record"
        print(json.dumps(out["gpu_tests"], indent=1))

In [ ]:
# Summary to paste back (aggregates only).
!python -m rbbd.cli sync --path env/$RBBD_SESSION_ID
print(json.dumps(SUMMARY, indent=1, default=str))
print("session hours:", round((time.time() - SESSION_T0) / 3600, 2))
!df -h /kaggle/working | tail -1
for NAME in CONFIGS:
    !python -m rbbd.cli status --config configs/{NAME}.yaml